In [66]:
import pandas as pd
import plotly.express as px
import numpy as np
import matplotlib.pyplot as plt
import plotly.io as pio

In [67]:
datapath = "data/"

df_history = pd.read_csv(f"{datapath}europe_yearly_full_release_long_format.csv")
df_prices = pd.read_csv(
    f"{datapath}european_wholesale_electricity_price_data_monthly.csv"
)

xls = pd.ExcelFile(f"{datapath}targets_download.xlsx")
print(f"found {len(xls.sheet_names)} sheets in targets file: {xls.sheet_names}")

found 4 sheets in targets file: ['raw_data_long', 'sources', 'capacity_target_wide', 'share_target_wide']


In [68]:
df_targets = pd.read_excel(
    f"{datapath}targets_download.xlsx", sheet_name="raw_data_long"
)

In [69]:
datasets = [df_history, df_targets, df_prices]

for df in datasets:
    print(f"Shape of df: {df.shape}")
    print(f"Available columns in df: {df.columns}")

Shape of df: (69310, 18)
Available columns in df: Index(['Area', 'ISO 3 code', 'Year', 'Area type', 'Continent', 'Ember region',
       'EU', 'OECD', 'G20', 'G7', 'ASEAN', 'Category', 'Subcategory',
       'Variable', 'Unit', 'Value', 'YoY absolute change', 'YoY % change'],
      dtype='str')
Shape of df: (2143, 8)
Available columns in df: Index(['COUNTRY_CODE', 'TARGET_YEAR', 'FUEL_CATEGORY', 'METRIC', 'VALUE',
       'SOURCE_ID', 'NUMBER_OF_UPDATES', 'COLLECTION_YEAR'],
      dtype='str')
Shape of df: (3801, 4)
Available columns in df: Index(['Country', 'ISO3 Code', 'Date', 'Price (EUR/MWhe)'], dtype='str')


In [70]:
# data transformations for df_history:
datapath = "data/"

df_history_2023 = (
    df_history.query("Year == 2023")
    .query("Category == 'Electricity generation'")
    .query("Unit == 'TWh'")
    .rename(columns={"ISO 3 code": "iso", "Variable": "source", "Value": "value"})
    .loc[:, ["iso", "source", "value"]]
)

df_history_2023.head()

,iso,source,value
1758,AUT,Clean,60.46
1759,AUT,Coal,0.00
1760,AUT,Fossil,10.82
1761,AUT,"Hydro, bioenergy and other renewables",45.23
1762,AUT,Renewables,60.46


In [71]:
# data transformations for df_targets:
df_targets_2030 = (
    df_targets.query("METRIC == 'generation_twh'")
    .query("TARGET_YEAR == 2030")
    .rename(
        columns={"COUNTRY_CODE": "iso", "FUEL_CATEGORY": "source", "VALUE": "value"}
    )
    .loc[:, ["iso", "source", "value"]]
)

df_targets_2030.head()

,iso,source,value
17,IRL,Solar,3.98
18,IRL,Wind,26.77
19,IRL,Bioenergy,1.00
20,IRL,Hydro,0.67
21,IRL,Other Renewables,0.41


In [72]:
df_gap = pd.merge(
    df_history_2023,
    df_targets_2030,
    on=["iso", "source"],
    how="inner",
    suffixes=["_history", "_target"],
)

df_gap = df_gap.assign(
    **{"gap_twh": lambda x: x["value_target"] - x["value_history"]}
).assign(**{"pct_missing": lambda x: x["gap_twh"] / x["value_target"] * 100})

df_gap

,iso,source,value_history,value_target,gap_twh,pct_missing
0,AUT,Fossil,10.82,6.0,-4.82,-80.333333
1,AUT,Fossil,10.82,6.0,-4.82,-80.333333
2,AUT,Wind,8.34,19.0,10.66,56.105263
3,AUT,Wind,8.34,19.0,10.66,56.105263
4,AUT,Wind,8.34,19.0,10.66,56.105263
...,...,...,...,...,...,...
269,GBR,Gas,101.66,16.0,-85.66,-535.375000
270,GBR,Gas,101.66,14.0,-87.66,-626.142857
271,GBR,Nuclear,40.60,38.0,-2.60,-6.842105
272,GBR,Nuclear,40.60,24.0,-16.60,-69.166667


In [73]:
# Create an Interactive Bar Chart
plot_source = "Solar"

df_graph = (
    df_gap.query(f"source == '{plot_source}'")
    .sort_values(by="value_target", ascending=True)
    .rename(columns={"iso": "country", "value_history": "current_twh"})
)

fig = px.bar(
    df_graph,
    y="country",
    x=["current_twh", "gap_twh"],
    title=f"<b>The {plot_source} Reality Gap (2023 vs 2030 Target)</b><br><i>Dark Blue = Current Generation, Light Blue = Remaining Gap to Target</i>",
    labels={"value": "Generation (TWh)", "country": "Country", "variable": "Metric"},
    orientation="h",  # Horizontal bars are easier to read for country names
    height=800,  # Tall chart to fit all countries
    color_discrete_map={
        "current_twh": "#1f77b4",
        "gap_twh": "#aec7e8",
    },  # Dark vs Light Blue
)

# Customizing the Layout for a Professional Look
fig.update_layout(
    xaxis_title="Electricity Generation (TWh)",
    legend_title_text="",
    showlegend=True,
    template="plotly_white",
    hovermode="y unified",  # Shows both values when hovering over a country
)

# Fix the legend names
new_names = {"current_twh": "Current (2023)", "gap_twh": "Remaining Gap"}
fig.for_each_trace(lambda t: t.update(name=new_names[t.name]))

fig.show()

## Prices

In [74]:
df_prices

,Country,ISO3 Code,Date,Price (EUR/MWhe)
0,Austria,AUT,2015-01-01,29.94
1,Belgium,BEL,2015-01-01,42.33
2,Czechia,CZE,2015-01-01,29.47
3,Denmark,DNK,2015-01-01,27.12
4,Estonia,EST,2015-01-01,33.84
...,...,...,...,...
3796,Slovakia,SVK,2026-02-01,122.36
3797,Slovenia,SVN,2026-02-01,124.98
3798,Spain,ESP,2026-02-01,20.37
3799,Sweden,SWE,2026-02-01,102.59


In [75]:
df_prices["Country"].unique()

<StringArray>
[        'Austria',         'Belgium',         'Czechia',         'Denmark',
         'Estonia',         'Finland',          'France',         'Germany',
          'Greece',         'Hungary',           'Italy',          'Latvia',
       'Lithuania',      'Luxembourg',     'Netherlands',          'Norway',
          'Poland',        'Portugal',         'Romania',        'Slovakia',
        'Slovenia',           'Spain',          'Sweden',     'Switzerland',
  'United Kingdom',        'Bulgaria',          'Serbia',         'Croatia',
         'Ireland',      'Montenegro', 'North Macedonia']
Length: 31, dtype: str

In [76]:
selected_countries = ["Germany", "Norway", "Austria", "Spain", "Italy"]

df_prices_filtered = (
    df_prices.query("Country in @selected_countries")
    .assign(**{"Date": lambda x: pd.to_datetime(x["Date"])})
    .sort_values(by="Date")
)

df_prices_filtered

,Country,ISO3 Code,Date,Price (EUR/MWhe)
0,Austria,AUT,2015-01-01,29.94
7,Germany,DEU,2015-01-01,29.94
10,Italy,ITA,2015-01-01,50.84
15,Norway,NOR,2015-01-01,29.19
21,Spain,ESP,2015-01-01,51.48
...,...,...,...,...
3791,Norway,NOR,2026-02-01,103.70
3773,Austria,AUT,2026-02-01,126.48
3782,Germany,DEU,2026-02-01,104.11
3786,Italy,ITA,2026-02-01,130.38


In [77]:
# --- 4.2 Create Interactive Line Chart ---
fig = px.line(
    df_prices_filtered,
    x="Date",
    y="Price (EUR/MWhe)",
    color="Country",
    title="<b>The Energy Crisis: Wholesale Electricity Prices (2015-2023)</b><br><i>Notice the divergence during the 2022 shock</i>",
    template="plotly_white",
    height=600,
)

# --- 4.3 Add Storytelling Elements ---
# Highlight the 2022 Crisis Year
fig.add_vrect(
    x0="2022-01-01",
    x1="2022-12-31",
    fillcolor="red",
    opacity=0.1,
    layer="below",
    line_width=0,
    annotation_text="2022 Energy Crisis",
    annotation_position="top left",
)

# Customize Layout
fig.update_layout(
    xaxis_title="",
    yaxis_title="Price (EUR/MWh)",
    legend_title_text="Country",
    hovermode="x unified",  # Shows all country prices for that month in one tooltip
)

fig.show()

### Correlation Analysis between Renewables rates and Wholesale Energy  in 2022

In [78]:
# calculate mean wholesale energy prices per year and country:
df_prices_2022_mean = (
    df_prices.rename(columns={"ISO3 Code": "iso", "Price (EUR/MWhe)": "price"})
    .assign(year=lambda d: pd.to_datetime(d["Date"]).dt.year)
    .groupby(["year", "iso"])["price"]
    .mean()
    .reset_index()
    .query("year == 2022")
)

print(f"Prices filtered yearly dataframe: \n {df_prices_2022_mean}")

Prices filtered yearly dataframe: 
      year  iso       price
195  2022  AUT  262.435000
196  2022  BEL  243.740000
197  2022  BGR  252.249167
198  2022  CHE  280.995833
199  2022  CZE  246.605000
200  2022  DEU  235.530833
201  2022  DNK  214.795833
202  2022  ESP  167.628333
203  2022  EST  191.971667
204  2022  FIN  153.479167
205  2022  FRA  275.061667
206  2022  GBR  239.596667
207  2022  GRC  279.389167
208  2022  HRV  271.685833
209  2022  HUN  270.901667
210  2022  IRL  225.170833
211  2022  ITA  303.088333
212  2022  LTU  229.226667
213  2022  LUX  235.530833
214  2022  LVA  225.913333
215  2022  NLD  241.204167
216  2022  NOR  138.802500
217  2022  POL  166.282500
218  2022  PRT  167.997500
219  2022  ROU  264.520000
220  2022  SRB  272.918333
221  2022  SVK  264.066667
222  2022  SVN  273.691667
223  2022  SWE  119.453333


In [79]:
# filter df_history for 2022 and for %
filter_keyword = "Renewables"

df_history_2022 = (
    df_history.query("Year == 2022")
    .query("Category == 'Electricity generation'")
    .query("Unit == '%'")
    .rename(columns={"ISO 3 code": "iso", "Variable": "source", "Value": "value"})
    .loc[:, ["iso", "source", "value"]]
    .query(f"source == '{filter_keyword}'")
)

df_history_2022.head()

,iso,source,value
1702,AUT,Renewables,78.18
3636,BEL,Renewables,25.90
4806,BIH,Renewables,32.05
6690,BGR,Renewables,19.15
8574,HRV,Renewables,63.48


In [80]:
df_prices_shares = pd.merge(
    df_history_2022,
    df_prices_2022_mean,
    on=["iso"],
    how="inner",
    suffixes=["_shares", "_prices"],
)
df_prices_shares = df_prices_shares.rename(columns={"value": "renewables_ratio"})

df_prices_shares

,iso,source,renewables_ratio,year,price
0,AUT,Renewables,78.18,2022,262.435000
1,BEL,Renewables,25.90,2022,243.740000
2,BGR,Renewables,19.15,2022,252.249167
3,HRV,Renewables,63.48,2022,271.685833
4,CZE,Renewables,12.86,2022,246.605000
5,DNK,Renewables,80.86,2022,214.795833
6,EST,Renewables,31.99,2022,191.971667
7,FIN,Renewables,53.57,2022,153.479167
8,FRA,Renewables,24.17,2022,275.061667
9,DEU,Renewables,44.37,2022,235.530833


In [81]:
fig = px.scatter(
    df_prices_shares,
    x="renewables_ratio",
    y="price",
    text="iso",  # Show country codes on dots
    size="price",  # Size bubbles by price impact
    color="renewables_ratio",
    title=f"<b>The Crisis Correlation (2022)</b><br><i>Higher {filter_keyword} (Right) vs Lower Prices (Bottom)</i>",
    trendline="ols",  # Adds a regression trendline (Ordinary Least Squares)
    template="plotly_white",
    color_continuous_scale="Bluered",
)

fig.update_traces(textposition="top center")

fig.update_layout(
    xaxis_title=f"Share of {filter_keyword} Power Generation 2022 (%)",
    yaxis_title="Average Wholesale Price 2022 (€/MWh)",
    height=600,
)

In [82]:
from scipy.stats import pearsonr

# Calculate Pearson r and p-value
r, p_value = pearsonr(df_prices_shares["renewables_ratio"], df_prices_shares["price"])

print(f"Pearson r: {r:.4f}")
print(f"P-value: {p_value:.4e}")

Pearson r: -0.3613
P-value: 5.4163e-02


### Carbon intensity leaderbord

In [83]:
print("Inspecting the values in the Columns 'Category', 'Subcategory' and 'Variable':")
print(f"{df_history["Category"].unique()}")
print(f"{df_history["Subcategory"].unique()}")
print(f"{df_history["Variable"].unique()}")

Inspecting the values in the Columns 'Category', 'Subcategory' and 'Variable':
<StringArray>
[    'Electricity demand', 'Electricity generation',    'Electricity imports',
 'Power sector emissions']
Length: 4, dtype: str
<StringArray>
[             'Demand',   'Demand per capita',      'Aggregate fuel',
                'Fuel',               'Total', 'Electricity imports',
       'CO2 intensity']
Length: 7, dtype: str
<StringArray>
[                               'Demand',
                     'Demand per capita',
                                 'Clean',
                                  'Coal',
                                'Fossil',
 'Hydro, bioenergy and other renewables',
                            'Renewables',
                                  'Wind',
                        'Wind and solar',
                             'Bioenergy',
                                   'Gas',
                             'Hard coal',
                                 'Hydro',
                   

In [84]:
df_history_CO2_2023 = (
    df_history.query("Year == 2023")
    .query("Variable == 'CO2 intensity'")
    .rename(columns={"ISO 3 code": "iso", "Value": "value", "Area": "Country"})
    .loc[:, ["iso", "Country", "value", "Unit"]]
    .sort_values(by="value", ascending=False)
)

df_history_CO2_2023

,iso,Country,value,Unit
32992,XKX,Kosovo,922.70,gCO2e per kWh
49187,POL,Poland,654.79,gCO2e per kWh
54831,SRB,Serbia,641.56,gCO2e per kWh
4898,BIH,Bosnia Herzegovina,612.33,gCO2e per kWh
45341,MKD,North Macedonia,567.29,gCO2e per kWh
10550,CYP,Cyprus,534.84,gCO2e per kWh
65499,TUR,Türkiye,493.73,gCO2e per kWh
40528,MLT,Malta,493.62,gCO2e per kWh
12452,CZE,Czechia,442.55,gCO2e per kWh
41569,MNE,Montenegro,407.58,gCO2e per kWh


In [85]:
fig = px.bar(
    df_history_CO2_2023,
    x="value",
    y="Country",
    orientation="h",
    title="<b>The 2023 Green Grid Leaderboard (gCO2/kWh)</b><br><i>Lower is Better. Note the gap between Hydro/Nuclear leaders and Coal-heavy grids.</i>",
    color="value",
    color_continuous_scale="RdYlGn_r",  # Red (Dirty) to Green (Clean) reversed
    text="value",
)

fig.update_layout(
    xaxis_title="Carbon Intensity (gCO2 per kWh)",
    yaxis_title="",
    coloraxis_showscale=False,
    height=600,
)

fig.update_traces(texttemplate="%{text:.0f}", textposition="outside")
fig.show()

In [123]:
# plots for germany: fossil vs renewables
# note: I am testing two different ways of filtering for keep_resources the df (query vs. lambda function)

# plausibility check: what's the sum of total generation. Looking at 2023
print(f"All energy sources listed in the data: {df_history_2023["source"].unique()}")
components = ["Coal", "Gas", "Nuclear", "Other fossil", "Solar", "Wind", "Other renewables", "Bioenergy", "Hydro"]
# hard coal + lignite == Coal ?

sum_components = df_history_2023.loc[lambda d: d["source"].isin(components), "value"].sum()
total = df_history_2023.loc[lambda d: d["source"].eq("Total generation"), "value"].sum()

print(f"(sum of components, total value, do they match?): {sum_components, total, np.isclose(sum_components, total)}")
print(f"The components I will look at are {components}.")


keep_resources = ["Coal", "Gas", "Nuclear", "Other fossil", "Solar", "Wind", "Other renewables", "Bioenergy", "Hydro"]

df_ger_generation_twh = (
    df_history
    .query("Area == 'Germany'")
    .query("Category == 'Electricity generation'")
    .query("Unit == 'TWh'")
    .rename(columns={"ISO 3 code": "iso", "Value": "value", "Area": "Country"})
    .loc[:, ["Year", "iso", "Country", "value", "Unit", "Variable"]]
    .assign(
        total=lambda d: d.loc[d["Variable"].eq("Total generation"), "value"].sum(),
        total_calculated=lambda d: d.loc[d["Variable"].isin(components), "value"].sum(),
    )
    .query("Variable in @keep_resources")
)

df_ger_generation_percent = (
    df_history
    .query("Area == 'Germany'")
    .query("Category == 'Electricity generation'")
    .query("Unit == '%'")
    .rename(columns={"ISO 3 code": "iso", "Value": "value", "Area": "Country"})
    .loc[:, ["Year", "iso", "Country", "value", "Unit", "Variable"]]
    .loc[lambda d: d["Variable"].isin(keep_resources)]
)

df_ger_generation_twh.head()

All energy sources listed in the data: <StringArray>
[                                'Clean',
                                  'Coal',
                                'Fossil',
 'Hydro, bioenergy and other renewables',
                            'Renewables',
                                  'Wind',
                        'Wind and solar',
                             'Bioenergy',
                                   'Gas',
                             'Hard coal',
                                 'Hydro',
                               'Lignite',
                               'Nuclear',
                          'Onshore wind',
                          'Other fossil',
                      'Other renewables',
                                 'Solar',
                      'Total generation',
                         'Offshore wind']
Length: 19, dtype: str
(sum of components, total value, do they match?): (np.float64(6315.48), np.float64(6315.4800000000005), np.True_)
The componen

,Year,iso,Country,value,Unit,Variable,total,total_calculated
22204,1990,DEU,Germany,310.04,TWh,Coal,20877.37,20877.37
22208,1990,DEU,Germany,0.07,TWh,Wind,20877.37,20877.37
22220,1990,DEU,Germany,1.60,TWh,Bioenergy,20877.37,20877.37
22221,1990,DEU,Germany,35.90,TWh,Gas,20877.37,20877.37
22223,1990,DEU,Germany,17.43,TWh,Hydro,20877.37,20877.37


In [124]:
fig = px.line(
    df_ger_generation_twh,
    x="Year",
    y="value",
    color="Variable",
    title="<b>Electricity generation in Germany in TWh</b><br><i>Historical evolution from 1990 to 2025</i>",
    template="plotly_white",
    height=600,
)

# Customize Layout
fig.update_layout(
    xaxis_title="",
    yaxis_title="Electricity generation (TWh)",
    legend_title_text="Resource",
    hovermode="x unified",  # Shows all country prices for that month in one tooltip
)

fig.show()